# EEG Similarity Metrics Analysis

This notebook provides a comprehensive analysis of EEG data quality and similarity metrics.

## 0. Path Setup

Add project source folder to Python path.

In [ ]:
import sys
from pathlib import Path

# Add src folder to Python path
project_root = Path.cwd().parent
src_path = project_root / 'src' / 'eeg_synthetic'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f'✓ Path setup complete: {src_path}')

## 1. Setup and Data Loading

Import required libraries and load the EEG dataset for analysis.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from scipy import signal, stats, linalg
from scipy.spatial.distance import jensenshannon

from data_loader import BCIAUTLoader, plot_normalized_arrays

In [ ]:
# Set up paths to the dataset
project_root = os.getcwd()
dataset_path = os.path.join(project_root, 'data')

# Initialize the data loader for BCI-AUT dataset
loader = BCIAUTLoader(dataset_path)

# Load training data for subject 3, session 3
# X_train_sess: EEG signals [trials, channels, time]
# y_train_sess: Labels for each trial
X_train_sess, y_train_sess, subjects_train_ids, session_train_ids = loader.get_data(
    subjects=[3], sessions=[3], modes=['Train']
)

# Load test data for the same subject and session
X_test_sess, y_test_sess, subjects_test_ids, session_test_ids = loader.get_data(
    subjects=[3], sessions=[3], modes=['Test']
)

# Visualize the normalized arrays to check data distribution
plot_normalized_arrays(X_train_sess, y_train_sess, X_test_sess, y_test_sess)

## 2. Synthetic Data Loading

Load and prepare synthetic EEG data for comparison with real data.

In [ ]:
def load_balanced_synthetic_limited(path_c0, path_c1, max_trials=1200):
    """
    Load balanced synthetic EEG data for both classes.
    
    Parameters:
    -----------
    path_c0 : str
        Path to class 0 synthetic data CSV file
    path_c1 : str
        Path to class 1 synthetic data CSV file
    max_trials : int
        Maximum number of trials to load per class
        
    Returns:
    --------
    X0_trial : torch.Tensor
        Class 0 EEG data with shape (trials, channels, time)
    y0_trial : torch.Tensor
        Class 0 labels
    X1_trial : torch.Tensor
        Class 1 EEG data with shape (trials, channels, time)
    y1_trial : torch.Tensor
        Class 1 labels
    """
    # Each trial has 8 channels, so multiply by 8 to get total rows
    rows_to_read = max_trials * 8

    def get_data_from_csv(path):
        """Helper function to read CSV and extract EEG data."""
        if not os.path.exists(path):
            print(f"ERROR: File not found - {path}")
            return None, None
        
        # Read CSV file with specified number of rows
        df = pd.read_csv(path, nrows=rows_to_read)
        
        # Extract time series columns (e.g., Time_0, Time_1, ..., Time_127)
        time_cols = [col for col in df.columns if col.startswith('Time')]
        X = df[time_cols].values
        
        # Extract condition labels (0 or 1)
        y = df['Condition'].values
        return X, y
    
    # Load data for both classes
    X0_rows, y0_rows = get_data_from_csv(path_c0)
    X1_rows, y1_rows = get_data_from_csv(path_c1)
    
    # Reshape from flat format to [trials, channels=8, time=128]
    X0_trial = torch.tensor(X0_rows).view(-1, 8, 128)
    y0_trial = torch.tensor(y0_rows[::8])  # Take every 8th label (one per trial)
    
    X1_trial = torch.tensor(X1_rows).view(-1, 8, 128)
    y1_trial = torch.tensor(y1_rows[::8])

    return X0_trial, y0_trial, X1_trial, y1_trial


# Paths to synthetic data CSV files
file_c0 = 'generated_samples/aegan_w_lae_data_c0_subj_3_sess_3.csv'
file_c1 = 'generated_samples/aegan_w_lae_data_c1_subj_3_sess_3.csv'

# Load up to 8600 trials per class
X0_syn, y0_syn, X1_syn, y1_syn = load_balanced_synthetic_limited(
    file_c0, file_c1, max_trials=8600
)

# Combine both classes into single tensors
X_syn_final = torch.cat([X0_syn, X1_syn], dim=0)
y_syn_final = torch.cat([y0_syn, y1_syn], dim=0)

## 3. Data Preprocessing

Apply filtering and baseline correction to prepare data for analysis.

In [ ]:
def mne_style_iir_filter(data, fs=128):
    """
    Apply bandpass IIR filter to EEG data.
    
    Parameters:
    -----------
    data : np.ndarray
        EEG data to filter
    fs : int
        Sampling frequency in Hz
        
    Returns:
    --------
    np.ndarray
        Filtered EEG data
    """
    # Define bandpass filter parameters
    low_cutoff = 0.1   # High-pass at 0.1 Hz (removes slow drifts)
    high_cutoff = 15.0  # Low-pass at 15 Hz (removes high-frequency noise)
    order = 4           # 4th order Butterworth filter
    
    # Design Butterworth bandpass filter
    b, a = signal.butter(order, [low_cutoff, high_cutoff], btype='bandpass', fs=fs)
    
    # Apply zero-phase filtering (forward and backward pass)
    return signal.filtfilt(b, a, data, axis=-1)


def apply_final_alignment(x_tensor):
    """
    Apply filtering and baseline correction to EEG data.
    
    Parameters:
    -----------
    x_tensor : torch.Tensor
        Input EEG data with shape (trials, channels, time)
        
    Returns:
    --------
    torch.Tensor
        Preprocessed EEG data with filtering and baseline correction applied
    """
    # Convert to numpy for signal processing
    x_np = x_tensor.detach().cpu().numpy()
    
    # Step 1: Apply bandpass filter (0.1-15 Hz)
    x_filtered = mne_style_iir_filter(x_np, fs=128)
    
    # Step 2: Baseline correction using first 26 samples (~200ms at 128 Hz)
    # This removes DC offset and normalizes the baseline period
    baseline_period = x_filtered[:, :, :26]
    baseline_mean = np.mean(baseline_period, axis=-1, keepdims=True)
    x_aligned = x_filtered - baseline_mean
    
    # Convert back to PyTorch tensor
    return torch.from_numpy(x_aligned.copy()).float()

In [ ]:
# ============================================================================
# PREPROCESS SYNTHETIC DATA
# ============================================================================

# Apply filtering and baseline correction to synthetic data
X_syn_ready = apply_final_alignment(X_syn_final)
X_syn_filtered = torch.from_numpy(X_syn_ready.detach().cpu().numpy().copy()).float()

# Z-score normalization (trial-wise)
# This standardizes each trial independently to have mean=0 and std=1
mu_syn = X_syn_filtered.mean(dim=(1, 2), keepdim=True)
std_syn = X_syn_filtered.std(dim=(1, 2), keepdim=True)
X_syn_filtered = (X_syn_filtered - mu_syn) / std_syn

# Permute dimensions from [trials, channels, time] to [trials, time, channels]
X_grouped_perm = X_syn_filtered.permute(0, 2, 1)

# Separate by class: 0 = "win" (non-target), 1 = "loss" (target)
syn_win = X_grouped_perm[y_syn_final == 0.0]
syn_loss = X_grouped_perm[y_syn_final == 1.0]

# ============================================================================
# PREPROCESS REAL DATA (TRAIN AND TEST)
# ============================================================================

# Convert real data to PyTorch tensors
X_train_tensor = torch.tensor(X_train_sess).float()
y_train_tensor = torch.tensor(y_train_sess).float()
X_test_sess_tensor = torch.tensor(X_test_sess).float()
y_test_sess_tensor = torch.tensor(y_test_sess).float()

# Apply the same filtering and baseline correction
X_train_tensor = apply_final_alignment(X_train_tensor)
X_test_sess_tensor = apply_final_alignment(X_test_sess_tensor)

# Z-score normalization for training data
mu_ref = X_train_tensor.mean(dim=(1, 2), keepdim=True)
std_ref = X_train_tensor.std(dim=(1, 2), keepdim=True)
X_train_real_z = (X_train_tensor - mu_ref) / std_ref

# Z-score normalization for test data
mu_ref_test = X_test_sess_tensor.mean(dim=(1, 2), keepdim=True)
std_ref_test = X_test_sess_tensor.std(dim=(1, 2), keepdim=True)
X_test_real_z = (X_test_sess_tensor - mu_ref_test) / std_ref_test

# Permute to [trials, time, channels] format
X_train_real_ready = X_train_real_z.permute(0, 2, 1)
X_test_real_ready = X_test_real_z.permute(0, 2, 1)

print(f"Real Data: Train {X_train_real_ready.shape}, Test {X_test_real_ready.shape}")

# Separate real data by class
real_train_0 = X_train_real_ready[y_train_tensor == 0]  # Non-target trials (train)
real_test_0 = X_test_real_ready[y_test_sess_tensor == 0]  # Non-target trials (test)
real_train_1 = X_train_real_ready[y_train_tensor == 1]  # Target trials (train)
real_test_1 = X_test_real_ready[y_test_sess_tensor == 1]  # Target trials (test)

## 4. Fréchet Inception Distance (FID)

Calculate FID using an autoencoder to measure similarity between real and synthetic data distributions.

In [ ]:
def calculate_fid_eeg(real_data, syn_data, encoder, device='cuda'):
    """
    Compute the Fréchet Inception Distance (FID) using an autoencoder as feature extractor.
    
    FID measures the distance between two multivariate Gaussians fitted to 
    the feature representations of real and synthetic data. Lower FID = better quality.
    
    Parameters:
    -----------
    real_data : Tensor
        Real EEG data with shape [N, Time, Channels]
    syn_data : Tensor
        Synthetic EEG data with shape [N, Time, Channels]
    encoder : nn.Module
        Trained autoencoder for feature extraction
    device : str
        Device to run computations on ('cuda' or 'cpu')
        
    Returns:
    --------
    float
        FID score (lower is better)
    """
    encoder.to(device)
    encoder.eval()
    
    def get_features(data):
        """Extract latent features using the encoder."""
        with torch.no_grad():
            # Convert to tensor if needed
            if isinstance(data, np.ndarray):
                data = torch.from_numpy(data).float()
            
            # Move to device and encode
            data = data.to(device).float()
            feat = encoder.encode(data.to(device))
            
            # Flatten features to 1D per sample
            feat = feat.reshape(feat.shape[0], -1).cpu().numpy()
        return feat
    
    # Extract features from both datasets
    act1 = get_features(real_data)
    act2 = get_features(syn_data)
    print(act1.shape, act2.shape)
    
    # Calculate mean and covariance for both distributions
    mu1, sigma1 = act1.mean(axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = act2.mean(axis=0), np.cov(act2, rowvar=False)

    # Compute FID using the formula:
    # FID = ||mu1 - mu2||^2 + Tr(sigma1 + sigma2 - 2*sqrt(sigma1*sigma2))
    diff = mu1 - mu2
    
    # Add small offset for numerical stability
    offset = np.eye(sigma1.shape[0]) * 1e-8
    covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))
    
    # Handle complex numbers from sqrtm
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    
    tr_covmean = np.trace(covmean)
    fid = diff.dot(diff) + np.trace(sigma1) + np.trace(sigma2) - 2 * tr_covmean
    
    return fid

In [ ]:
# ============================================================================
# LOAD PRETRAINED AUTOENCODER FOR FID CALCULATION
# ============================================================================

from eeggan.nn_architecture.ae_networks import TransformerAutoencoder

# Define target dimension constants
TARGET_CHANNELS = 0      # Compress along channel dimension
TARGET_TIMESERIES = 1    # Compress along time dimension

# Load the pretrained autoencoder checkpoint
checkpoint = torch.load(
    'trained_ae/my_autoencoder_subj_all_sess_all.pt', 
    map_location='cpu', 
    weights_only=False
)

# Extract configuration from checkpoint
config = checkpoint['configuration']

# Determine which dimension to compress
target_value = config['target']
if target_value == 'time':
    target_value = TARGET_TIMESERIES
elif target_value == 'channels':
    target_value = TARGET_CHANNELS

# Initialize the Transformer Autoencoder with saved configuration
ae_model = TransformerAutoencoder(
    input_dim=config['input_dim'],          # Input dimension size
    output_dim=config['output_dim'],        # Compressed dimension size
    output_dim_2=config['output_dim_2'],    # Secondary output dimension
    target=target_value,                     # Which dimension to compress
    hidden_dim=config['hidden_dim'],        # Hidden layer size
    num_layers=config['num_layers'],        # Number of transformer layers
    num_heads=config['num_heads']           # Number of attention heads
)

# Load the trained weights
ae_model.load_state_dict(checkpoint['model'])

# Set to evaluation mode (disables dropout, etc.)
ae_model.eval()

# Set target constants on the model
ae_model.TARGET_TIMESERIES = TARGET_TIMESERIES
ae_model.TARGET_CHANNELS = TARGET_CHANNELS

## 5. Comprehensive Similarity Metrics

This section computes multiple statistical metrics to evaluate the quality of synthetic EEG data:

**Distance Metrics:**
- **RMSE (Root Mean Square Error)**: Measures point-wise differences between signals
- **Jensen-Shannon Divergence**: Measures similarity between probability distributions
- **Wasserstein Distance**: Measures the "earth mover's distance" between distributions
- **Spectral Similarity**: Compares power spectral densities using correlation

**Intrinsic Properties:**
- **Variance**: Measures signal variability
- **Spatial Correlation**: Measures inter-channel dependencies
- **Self-Similarity**: Measures diversity within the dataset

In [ ]:
def calculate_eeg_stats(real_tensor, syn_tensor, fs=128):
    """
    Compute comprehensive similarity metrics between real and synthetic EEG data.
    
    This function calculates four key metrics:
    1. RMSE: Point-wise signal differences
    2. Jensen-Shannon Divergence: Distribution similarity
    3. Wasserstein Distance: Optimal transport distance
    4. Spectral Similarity: Frequency domain correlation
    
    Parameters:
    -----------
    real_tensor : torch.Tensor
        Real EEG data with shape [Batch, Time, Channels]
    syn_tensor : torch.Tensor
        Synthetic EEG data with shape [Batch, Time, Channels]
    fs : int
        Sampling frequency in Hz (default: 128)
        
    Returns:
    --------
    dict
        Dictionary containing mean values for each metric
    """
    # Convert tensors to numpy arrays for scipy operations
    r = real_tensor.detach().cpu().numpy()
    s = syn_tensor.detach().cpu().numpy()
    
    # Use the minimum batch size to handle unequal dataset sizes
    batch_size = min(r.shape[0], s.shape[0])
    num_channels = r.shape[2]
    
    # Initialize lists to store per-trial metrics
    trial_rmse, trial_js, trial_wd, trial_psd_sim = [], [], [], []

    # ========================================================================
    # COMPUTE POWER SPECTRAL DENSITY (PSD) FOR ALL TRIALS
    # ========================================================================
    # Welch's method: estimates power spectral density using overlapping segments
    # Transpose to [Batch, Channels, Time] format required by welch
    # nperseg=128: use full epoch length for frequency resolution
    _, p_r = signal.welch(r.transpose(0, 2, 1), fs=128, axis=-1, nperseg=128)
    _, p_s = signal.welch(s.transpose(0, 2, 1), fs=128, axis=-1, nperseg=128)

    # ========================================================================
    # ITERATE THROUGH EACH TRIAL
    # ========================================================================
    for i in range(batch_size):
        # Initialize lists for per-channel metrics within this trial
        ch_rmse, ch_js, ch_wd, ch_psd = [], [], [], []
        
        # ====================================================================
        # ITERATE THROUGH EACH CHANNEL
        # ====================================================================
        for ch in range(num_channels):
            # ----------------------------------------------------------------
            # METRIC 1: RMSE (Root Mean Square Error)
            # ----------------------------------------------------------------
            # Measures average point-wise difference between signals
            # Lower RMSE = more similar signals
            mse = np.mean((r[i, :, ch] - s[i, :, ch])**2)
            ch_rmse.append(np.sqrt(mse))
            
            # ----------------------------------------------------------------
            # METRIC 2: Wasserstein Distance (Earth Mover's Distance)
            # ----------------------------------------------------------------
            # Measures the minimum "cost" to transform one distribution into another
            # Robust to outliers, captures shape differences
            ch_wd.append(stats.wasserstein_distance(r[i, :, ch], s[i, :, ch]))
            
            # ----------------------------------------------------------------
            # METRIC 3: Jensen-Shannon Divergence
            # ----------------------------------------------------------------
            # Symmetric measure of similarity between two probability distributions
            # Range: [0, 1], where 0 = identical distributions
            
            # Determine common histogram range for fair comparison
            h_range = (min(r[i,:,ch].min(), s[i,:,ch].min()), 
                      max(r[i,:,ch].max(), s[i,:,ch].max()))
            
            # Create normalized histograms (probability distributions)
            p = np.histogram(r[i, :, ch], bins=100, range=h_range, density=True)[0]
            q = np.histogram(s[i, :, ch], bins=100, range=h_range, density=True)[0]
            
            # Normalize to ensure they sum to 1 (add epsilon for numerical stability)
            p = p / (p.sum() + 1e-10)
            q = q / (q.sum() + 1e-10)
            
            # Compute JS divergence
            ch_js.append(jensenshannon(p, q))
            
            # ----------------------------------------------------------------
            # METRIC 4: Spectral Similarity (PSD Correlation)
            # ----------------------------------------------------------------
            # Compares frequency content using Pearson correlation
            # Range: [-1, 1], where 1 = perfect positive correlation
            ps = np.corrcoef(p_r[i, ch, :], p_s[i, ch, :])[0, 1]
            
            # Only include valid correlations (skip NaN values)
            if not np.isnan(ps): 
                ch_psd.append(ps)
        
        # Average metrics across all channels for this trial
        trial_rmse.append(np.mean(ch_rmse))
        trial_js.append(np.mean(ch_js))
        trial_wd.append(np.mean(ch_wd))
        trial_psd_sim.append(np.mean(ch_psd))

    # Return average metrics across all trials
    return {
        "RMSE": np.mean(trial_rmse),
        "JS_Divergence": np.mean(trial_js),
        "Wasserstein_Dist": np.mean(trial_wd),
        "Spectral_Sim": np.mean(trial_psd_sim)
    }


# ============================================================================
# FUNCTION: CALCULATE INTRINSIC PROPERTIES
# ============================================================================
def calculate_intrinsic_properties(tensor, num_samples_sim=500):
    """
    Compute intrinsic statistical properties of EEG data.
    
    These metrics characterize the data itself, not comparisons between datasets:
    1. Variance: Signal variability over time
    2. Spatial Correlation: Inter-channel dependencies
    3. Self-Similarity: Diversity/consistency within the dataset
    
    Parameters:
    -----------
    tensor : torch.Tensor
        EEG data with shape [Batch, Time, Channels]
    num_samples_sim : int
        Number of samples to use for self-similarity calculation (default: 500)
        
    Returns:
    --------
    dict
        Dictionary containing the three intrinsic metrics
    """
    # Convert to numpy for calculations
    data = tensor.detach().cpu().numpy()
    batch_size = data.shape[0]
    num_channels = data.shape[2]
    
    # Initialize lists for per-trial metrics
    vars_per_trial = []
    spat_corr_per_trial = []

    # ========================================================================
    # COMPUTE VARIANCE AND SPATIAL CORRELATION (TRIAL-BY-TRIAL)
    # ========================================================================
    for i in range(batch_size):
        # --------------------------------------------------------------------
        # METRIC 1: Variance
        # --------------------------------------------------------------------
        # Measures how much the signal fluctuates over time
        # Higher variance = more dynamic signals
        # Compute variance along time axis, then average across channels
        vars_per_trial.append(np.mean(np.var(data[i, :, :], axis=0)))
        
        # --------------------------------------------------------------------
        # METRIC 2: Spatial Correlation (Inter-channel correlation)
        # --------------------------------------------------------------------
        # Measures how similar different channels are to each other
        # High correlation = channels carry redundant information
        if num_channels > 1:
            # Compute correlation matrix between all channel pairs
            # Transpose so channels are rows (required by corrcoef)
            matrix = np.corrcoef(data[i, :, :].T)
            
            # Extract upper triangle (avoid diagonal and duplicates)
            triu_idx = np.triu_indices(num_channels, k=1)
            
            # Average correlation across all channel pairs
            spat_corr_per_trial.append(np.nanmean(matrix[triu_idx]))

    # ========================================================================
    # COMPUTE SELF-SIMILARITY (DATASET DIVERSITY)
    # ========================================================================
    # Measures how similar trials are to each other within the dataset
    # Low self-similarity = high diversity (good for generalization)
    # High self-similarity = repetitive/homogeneous data
    
    # Flatten each trial to a 1D vector [Time*Channels]
    flat_data = data.reshape(batch_size, -1)
    
    # Subsample if dataset is too large (for computational efficiency)
    if batch_size > num_samples_sim:
        idx = np.random.choice(batch_size, num_samples_sim, replace=False)
        flat_data = flat_data[idx]
    
    # Compute correlation matrix between all trial pairs
    dist_matrix = np.corrcoef(flat_data)
    
    # Extract upper triangle (avoid diagonal self-correlations)
    triu_idx_sim = np.triu_indices(dist_matrix.shape[0], k=1)
    
    # Average correlation across all trial pairs
    self_sim = np.nanmean(dist_matrix[triu_idx_sim])

    return {
        "Variance": np.mean(vars_per_trial),
        "SpatialCorr": np.mean(spat_corr_per_trial) if spat_corr_per_trial else 0.0,
        "SelfSimilarity": self_sim
    }


# ============================================================================
# EVALUATION SETUP: DEFINE COMPARISONS
# ============================================================================
# Create list of comparison scenarios to evaluate
# Each tuple: (name, real_class0, syn_class0, real_class1, syn_class1)
comparisons = [
    ("Syn vs Train", real_train_0, syn_win, real_train_1, syn_loss),  # Training set comparison
    ("Syn vs Test",  real_test_0,  syn_win,  real_test_1,  syn_loss),  # Test set comparison
    ("Benchmark",    real_test_0,  real_train_0, real_test_1, real_train_1)  # Real train vs test baseline
]

# Initialize results dictionary
results_comp = {comp[0]: {"Win": {}, "Loss": {}} for comp in comparisons}

# ============================================================================
# COMPUTE COMPARISON METRICS FOR ALL SCENARIOS
# ============================================================================
for name, r0, s0, r1, s1 in comparisons:
    # ------------------------------------------------------------------------
    # Evaluate Class 0 (Win/Non-Target)
    # ------------------------------------------------------------------------
    fid_win = calculate_fid_eeg(r0, s0, ae_model)  # FID using autoencoder
    stats_win = calculate_eeg_stats(r0, s0)         # Statistical metrics
    results_comp[name]["Win"] = {"FID": fid_win, **stats_win}
    
    # ------------------------------------------------------------------------
    # Evaluate Class 1 (Loss/Target)
    # ------------------------------------------------------------------------
    fid_loss = calculate_fid_eeg(r1, s1, ae_model)  # FID using autoencoder
    stats_loss = calculate_eeg_stats(r1, s1)         # Statistical metrics
    results_comp[name]["Loss"] = {"FID": fid_loss, **stats_loss}

# ============================================================================
# COMPUTE INTRINSIC PROPERTIES FOR ALL DATASETS
# ============================================================================
# Define datasets to analyze
intrinsic_data = {
    "Real Train": {"Win": real_train_0, "Loss": real_train_1},
    "Real Test":  {"Win": real_test_0,  "Loss": real_test_1},
    "Synthetic":  {"Win": syn_win,       "Loss": syn_loss}
}

# Initialize results dictionary
intrinsic_results = {k: {} for k in intrinsic_data.keys()}

# Compute intrinsic properties for each dataset and class
for key, datasets in intrinsic_data.items():
    intrinsic_results[key]["Win"] = calculate_intrinsic_properties(datasets["Win"])
    intrinsic_results[key]["Loss"] = calculate_intrinsic_properties(datasets["Loss"])

# ============================================================================
# DISPLAY RESULTS: TABLE 1 - COMPARISON METRICS
# ============================================================================
metrics_comp = ["FID", "RMSE", "JS_Divergence", "Wasserstein_Dist", "Spectral_Sim"]

print("\n" + "="*110)
print(f"{'TABLE 1: COMPARISON METRICS (Distances and Similarities)':^110}")
print("="*110)
print(f"{'COMPARISON':<20} | {'METRIC':<20} | {'WIN (CLASS 0)':<20} | {'LOSS (CLASS 1)':<20}")
print("-" * 110)

for comp in results_comp:
    for i, m in enumerate(metrics_comp):
        # Only print comparison name on first row
        row_label = comp if i == 0 else ""
        
        # Get metric values for both classes
        v_win = results_comp[comp]["Win"][m]
        v_loss = results_comp[comp]["Loss"][m]
        
        # Print formatted row
        print(f"{row_label:<20} | {m:<20} | {v_win:>20.4f} | {v_loss:>20.4f}")
    print("-" * 110)

# ============================================================================
# DISPLAY RESULTS: TABLE 2 - INTRINSIC PROPERTIES
# ============================================================================
intrinsic_metrics = ["Variance", "SpatialCorr", "SelfSimilarity"]

print("\n" + "="*110)
print(f"{'TABLE 2: INTRINSIC PROPERTIES OF DATASETS':^110}")
print("="*110)
print(f"{'DATASET':<20} | {'METRIC':<20} | {'WIN (CLASS 0)':<20} | {'LOSS (CLASS 1)':<20}")
print("-" * 110)

for ds_name in intrinsic_results:
    for i, m in enumerate(intrinsic_metrics):
        # Only print dataset name on first row
        row_label = ds_name if i == 0 else ""
        
        # Get metric values for both classes
        v_win = intrinsic_results[ds_name]["Win"][m]
        v_loss = intrinsic_results[ds_name]["Loss"][m]
        
        # Print formatted row
        print(f"{row_label:<20} | {m:<20} | {v_win:>20.4f} | {v_loss:>20.4f}")
    print("-" * 110)
